# Scénario 11c : les sorties du retard, MLP face à CatBoost

Journal : [`tableau_resultats.md`](tableau_resultats.md). Essai **bonus et indépendant** des autres branches de machine learning du projet.

**Règle visée pour estimer les `PAY_1` = 1** (décision de l'utilisateur) :
- un client **sain en août** reste sain en septembre (la persistance domine : 95,8 % des sains d'août le restent ; l'arrivée d'un retard n'est pas prévisible a priori) ;
- un client **en retard en août** : un modèle décide s'il **sort du retard** ; sinon, son retard est maintenu.

Ce notebook entraîne et juge ce modèle des sorties du retard. **Le MLP est le modèle retenu, CatBoost son challenger** : à précision égale, celui qui retrouve le plus de sorties l'emporte.

**Données** : `data/creditcard_pret_ingestion.csv` (niveau 0), sans `dpnm`, sans `ID`, sans données personnelles, **sans correctifs** (codifications d'origine, comme le `11b`). Même découpage de septembre que les notebooks 01 à 11b (clients dont `PAY_1` n'est pas un 1, 80/20, `stratify` sur `PAY_1`, `random_state=42`) ; le test n'est pas lu.

**Exemples** : les clients **en retard le mois précédent** (codification 2 et plus), pour trois mois cibles : **septembre**, **août** et **juillet**. Un client en retard en juin qui sort en juillet est un exemple du même type qu'un client en retard en août qui sort en septembre. L'empilement s'arrête à juillet : à partir de juillet, la banque applique une pratique stable (passage 18 du journal) ; la limite est fixée par cette logique métier, pas par les résultats. Les variables sont celles du `11a`, rangées par ancienneté par rapport au mois cible (`codif_m1` = mois précédent, `solde_m0` = solde du mois, …) ; **aucune colonne ajoutée**. Les modèles sont notés **sur septembre seulement**.

**Cible** : 1 = **sortie du retard** (codification du mois saine : -2, -1 ou 0) ; 0 = retard maintenu (2 et plus).

**Méthode** :
- boucle d'optimisation de l'essai (fenêtre de trois valeurs autour du réglage courant, 10 tours au plus, surapprentissage limité à 7 % relatif, alerte si l'écart-type des plis dépasse 2 points), départ aux meilleurs réglages du `11b` ;
- **critère : la précision moyenne sur les sorties** (aire sous la courbe précision-rappel) : elle juge la capacité à classer les vraies sorties en tête, sur tous les seuils ;
- **MLP stabilisé** : moyenne des probabilités de 5 MLP entraînés avec des graines différentes (ses résultats dépendent du tirage de ses poids de départ) ;
- **seuil de décision : précision marginale de 80 %**, fixée à l'avance par l'utilisateur. Les clients sont rangés du plus probablement sorti au moins probable ; on annonce des sorties **tant que, parmi les 50 derniers annoncés, au moins 80 % sont de vraies sorties** (au moins 4 sorties annoncées sur 5 sont vraies, même en bout de liste). Le seuil est calculé sur les plis d'entraînement (validation croisée interne), puis jugé sur le pli de validation, qu'il n'a pas vu. Les niveaux 70 % et 90 % sont affichés **pour information seulement** : le niveau retenu reste 80 %.

**Limites** :
- les `PAY_1` = 1 sont les clients que la banque n'a pas tranchés : la précision mesurée sur les clients tranchés n'est pas garantie pour eux, et elle ne se vérifie pas sans le paiement d'octobre ;
- le modèle apprend la codification habituelle de la banque : si `PAY_1` est moins fiable en septembre que les autres mois (étude de fiabilité à mener), le modèle apprend aussi ces erreurs. L'estimation des 1 se garde donc **à côté** de `PAY_1`, jamais à sa place.

## 1. Données et découpage de septembre (identique au 11b)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
ORIGINE = PAY[1:] + ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
assert sorted(ORIGINE + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)

mask_1 = df["PAY_1"] == 1
connus, a_trancher = df[~mask_1], df[mask_1]
X_train, X_test, y_train, y_test = train_test_split(
    connus[ORIGINE], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}, non lu) ; PAY_1 = 1 à trancher : {len(a_trancher)}")
print(f"PAY_1 = 1 en retard en août (PAY_2 >= 2) : {int((a_trancher['PAY_2'] >= 2).sum())} ; sains en août : {int((a_trancher['PAY_2'] <= 0).sum())} ; "
      f"1 aussi en août : {int((a_trancher['PAY_2'] == 1).sum())}")

PAY_1 connus : 26312 (train 21049, test 5263, non lu) ; PAY_1 = 1 à trancher : 3688
PAY_1 = 1 en retard en août (PAY_2 >= 2) : 1824 ; sains en août : 1836 ; 1 aussi en août : 28


## 2. Les exemples : clients en retard le mois précédent (septembre, août, juillet)

Pour chaque mois cible, ce que la banque avait sous les yeux à la fin du mois : les codifications des mois précédents (`codif_m1` = mois précédent, regroupées en -2, -1, 0, 2 et plus ; vide avant avril), les soldes et paiements du mois (`m0`) et des mois précédents, le plafond. Septembre : clients du train seulement ; août et juillet : clients du train et clients codifiés 1 en septembre (leur codification d'août ou de juillet est tranchée), comme dans le `11a`. Aucun client du test. On garde les exemples **en retard le mois précédent** dont la codification du mois est tranchée (pas un 1).

In [2]:
MOIS_NOMS = {1: "septembre", 2: "août", 3: "juillet"}

def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 (et 1) inchangés ; un mois absent (vide) reste vide
    return codifications.mask(codifications >= 2, 2)

def exemples_du_mois(clients, n):
    # Ce que la banque voyait à la fin du mois n, en colonnes relatives (m1 = mois précédent) ; vide si le mois est antérieur à avril
    d = df.loc[clients]
    bloc = pd.DataFrame(index=d.index)
    for k in range(1, 6):
        bloc[f"codif_m{k}"] = regrouper(d[f"PAY_{n + k}"]) if n + k <= 6 else np.nan
    for k in range(0, 6):
        bloc[f"solde_m{k}"] = d[f"BILL_AMT{n + k}"] if n + k <= 6 else np.nan
        bloc[f"paiement_m{k}"] = d[f"PAY_AMT{n + k}"] if n + k <= 6 else np.nan
    bloc["LIMIT_BAL"] = d["LIMIT_BAL"]
    bloc["codification_du_mois"] = d[f"PAY_{n}"]
    bloc["mois"] = n
    bloc["client"] = d.index
    return bloc

clients_sept, clients_1 = X_train.index, a_trancher.index
emp = pd.concat([exemples_du_mois(clients_sept, 1)]
                + [exemples_du_mois(clients_sept.union(clients_1), n) for n in (2, 3)], ignore_index=True)
emp = emp[(emp["codif_m1"] == 2) & (emp["codification_du_mois"] != 1)].reset_index(drop=True)   # en retard le mois précédent, mois tranché
emp["sortie"] = (emp["codification_du_mois"] <= 0).astype(int)
assert not emp["client"].isin(X_test.index).any(), "un client du test est dans les exemples"
assert (emp["codif_m5"].notna() == (emp["mois"] == 1)).all(), "codif_m5 renseignée <=> septembre"

CATEGORIELLES = [f"codif_m{k}" for k in range(1, 6)]
NUMERIQUES = [f"solde_m{k}" for k in range(0, 6)] + [f"paiement_m{k}" for k in range(0, 6)] + ["LIMIT_BAL"]
VARIABLES = CATEGORIELLES + NUMERIQUES
X_emp, y_emp = emp[VARIABLES], emp["sortie"]
sept = np.flatnonzero(emp["mois"].to_numpy() == 1)
assert len(sept) == int((X_train["PAY_2"] >= 2).sum())

tableau = emp.groupby(emp["mois"].map(MOIS_NOMS).rename("Mois cible")).agg(
    Exemples=("sortie", "size"), Sorties=("sortie", "sum"), Clients=("client", "nunique"))
tableau["Retards maintenus"] = tableau["Exemples"] - tableau["Sorties"]
tableau["Part de sorties (%)"] = (tableau["Sorties"] / tableau["Exemples"] * 100).round(1)
tableau = tableau.reindex(list(MOIS_NOMS.values()))
tableau.loc["Total"] = [tableau["Exemples"].sum(), tableau["Sorties"].sum(), emp["client"].nunique(),
                        tableau["Retards maintenus"].sum(), round(emp["sortie"].mean() * 100, 1)]
display(tableau)

,Exemples,Sorties,Clients,Retards maintenus,Part de sorties (%)
Mois cible,,,,,
septembre,2071.0,366.0,2071.0,1705.0,17.7
août,3597.0,1057.0,3597.0,2540.0,29.4
juillet,2985.0,762.0,2985.0,2223.0,25.5
Total,8653.0,2185.0,4867.0,6468.0,25.3


**Comment lire** : une ligne par mois cible ; les exemples sont des clients en retard le mois précédent, partagés entre sorties du retard et retards maintenus. Seule la ligne septembre sert à noter les modèles ; août et juillet ajoutent des exemples d'apprentissage.

## 3. Plis, critère et modèles

**Plis** : 5 plis par client, stratifiés sur les sorties de septembre (`random_state=42`). Chaque pli est noté sur le septembre de ses clients, et ces clients sont retirés de l'apprentissage du pli pour tous leurs mois.

**Critère** : la précision moyenne sur les sorties, calculée sur septembre ; le score en train porte lui aussi sur septembre (même mesure des deux côtés pour le contrôle du surapprentissage).

**Modèles** :
- **MLP** (retenu) : moyenne des probabilités de 5 MLP de graines différentes ; taille des couches cachées et régularisation `alpha` explorées ; arrêt anticipé (`early_stopping`) ;
- **CatBoost** (challenger) : réglé comme dans l'essai (codifications en une colonne par valeur, tous les cœurs, 1 000 itérations, perte `Logloss`) ; profondeur, `learning_rate` et `l2_leaf_reg` explorés ; poids des classes `Balanced` ou `SqrtBalanced`.

Départ de la boucle aux meilleurs réglages du `11b`.

In [3]:
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points
MAX_TOURS = 10               # tours de la boucle, par modèle

import time
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

def score_septembre(estimateur, X, y):
    # Précision moyenne sur les sorties, sur les seuls exemples de septembre (codif_m5 renseignée)
    septembre = X["codif_m5"].notna().to_numpy()
    return average_precision_score(np.asarray(y)[septembre], estimateur.predict_proba(X[septembre])[:, 1])

# Logarithme signé des soldes, des paiements et du plafond (un vide reste vide)
def log_montants(X):
    X = X.copy()
    X[NUMERIQUES] = np.sign(X[NUMERIQUES]) * np.log1p(np.abs(X[NUMERIQUES]))
    return X

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur (un mois absent est une valeur à part) ;
    # nombres : un vide est remplacé par 0 et signalé par un indicateur, puis mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", make_pipeline(SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), StandardScaler()), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (CatBoost traite les vides)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

class MoyenneGraines(ClassifierMixin, BaseEstimator):
    # Moyenne des probabilités de plusieurs copies du même modèle, chacune avec sa graine
    def __init__(self, modele=None, graines=5, parametre_graine="mlpclassifier__random_state"):
        self.modele, self.graines, self.parametre_graine = modele, graines, parametre_graine

    def fit(self, X, y):
        self.modeles_ = [clone(self.modele).set_params(**{self.parametre_graine: g}).fit(X, y) for g in range(self.graines)]
        self.classes_ = self.modeles_[0].classes_
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.modeles_], axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

# Plis par client, notés sur septembre
clients_emp = emp["client"].to_numpy()
cv = []
for _, val in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(sept, y_emp.to_numpy()[sept]):
    clients_val = np.unique(clients_emp[sept[val]])
    cv.append((np.flatnonzero(~np.isin(clients_emp, clients_val)), sept[val]))
print(f"Plis : apprentissage de {min(len(a) for a, _ in cv)} à {max(len(a) for a, _ in cv)} exemples (trois mois), "
      f"validation de {min(len(v) for _, v in cv)} à {max(len(v) for _, v in cv)} clients (septembre)")

ESPACES = {
    "MLP": {
        "modele": MoyenneGraines(make_pipeline(FunctionTransformer(log_montants), preprocesseur_lineaire(),
                                               MLPClassifier(max_iter=500, early_stopping=True, random_state=0))),
        "echelles": {"modele__mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "modele__mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"modele__mlpclassifier__hidden_layer_sizes": (256, 128), "modele__mlpclassifier__alpha": 1e-3},   # meilleur réglage du 11b
        "choix": {},
    },
    "CatBoost": {
        "modele": make_pipeline(FunctionTransformer(log_montants), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="Logloss", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 4, "catboostclassifier__learning_rate": 0.03, "catboostclassifier__l2_leaf_reg": 10},   # meilleur réglage du 11b
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES), f"; précision moyenne d'un tri au hasard (part de sorties en septembre) : {y_emp.to_numpy()[sept].mean() * 100:.1f} %")

Plis : apprentissage de 7688 à 7719 exemples (trois mois), validation de 414 à 415 clients (septembre)
Modèles : ['MLP', 'CatBoost'] ; précision moyenne d'un tri au hasard (part de sorties en septembre) : 17.7 %


## 4. La boucle d'optimisation

Pour chaque modèle, tour après tour : GridSearch sur les 5 plis par client, puis choix du meilleur réglage **parmi ceux qui surapprennent de 7 % relatif au plus**. Une ligne à la fin de chaque tour, puis une ligne par modèle : son meilleur réglage admis, sa précision moyenne en validation croisée et la raison de l'arrêt.

In [4]:
def fenetre(echelle, position):
    # La valeur courante et ses deux voisines dans l'échelle
    return echelle[max(0, position - 1): position + 2]

def libelle(params):
    return ", ".join(f"{k.split('__')[-1]}={v}" for k, v in params.items())

def optimiser(nom, espace):
    positions = {p: espace["echelles"][p].index(v) for p, v in espace["depart"].items()}
    lignes, raison, debut = [], f"{MAX_TOURS} tours atteints", time.perf_counter()
    for tour in range(1, MAX_TOURS + 1):
        debut_tour = time.perf_counter()
        grille = {p: fenetre(espace["echelles"][p], i) for p, i in positions.items()} | espace["choix"]
        recherche = GridSearchCV(espace["modele"], grille, scoring=score_septembre, cv=cv, n_jobs=-1, return_train_score=True, refit=False)
        recherche.fit(X_emp, y_emp)
        r = recherche.cv_results_
        res = pd.DataFrame({"tour": tour, "params": r["params"], "validation": r["mean_test_score"] * 100,
                            "écart-type des plis": r["std_test_score"] * 100, "train": r["mean_train_score"] * 100})
        res["surapprentissage (% relatif)"] = (res["train"] - res["validation"]) / res["train"] * 100
        res["admis"] = res["surapprentissage (% relatif)"] <= SURAPPRENTISSAGE_MAX
        lignes.append(res)
        admis = res[res["admis"]]
        if admis.empty:
            print(f"{nom}, tour {tour} : {len(res)} réglages, aucun admis ({round(time.perf_counter() - debut_tour)} s)", flush=True)
            raison = f"tour {tour} : aucun réglage sous {SURAPPRENTISSAGE_MAX} % de surapprentissage"
            break
        meilleur = admis.loc[admis["validation"].idxmax(), "params"]
        print(f"{nom}, tour {tour} : {len(res)} réglages, {len(admis)} admis ; meilleur admis {admis['validation'].max():.2f} % "
              f"({libelle(meilleur)}) ({round(time.perf_counter() - debut_tour)} s)", flush=True)
        nouvelles = {p: espace["echelles"][p].index(meilleur[p]) for p in positions}
        if nouvelles == positions:
            raison = f"tour {tour} : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle)"
            break
        positions = nouvelles
    return pd.concat(lignes, ignore_index=True), raison, round(time.perf_counter() - debut)

resultats, lignes, meilleurs = {}, [], {}
for nom, espace in ESPACES.items():
    historique, raison, duree = optimiser(nom, espace)
    resultats[nom] = {"historique": historique, "raison": raison, "duree": duree}
    admis = historique[historique["admis"]]
    if admis.empty:
        ligne, alerte = historique.loc[historique["surapprentissage (% relatif)"].idxmin()], "⚠️ aucun réglage admis (le moins surappris est affiché)"
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
    meilleurs[nom] = ligne["params"]
    print(f"{nom} : {ligne['validation']:.2f} % ± {ligne['écart-type des plis']:.2f} ({libelle(ligne['params'])}) ; arrêt : {raison} ({duree} s)")
    lignes.append({"Modèle": nom, "Réglage": libelle(ligne["params"]),
                   "Précision moyenne sur les sorties, validation (%)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2), "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Tours": int(historique["tour"].max()), "Arrêt": raison, "Durée (s)": duree})
display(pd.DataFrame(lignes).set_index("Modèle"))

MLP, tour 1 : 9 réglages, 9 admis ; meilleur admis 88.15 % (alpha=0.0001, hidden_layer_sizes=(256, 128, 64)) (24 s)
MLP, tour 2 : 6 réglages, 6 admis ; meilleur admis 88.15 % (alpha=0.0001, hidden_layer_sizes=(256, 128, 64)) (19 s)
MLP : 88.15 % ± 1.27 (alpha=0.0001, hidden_layer_sizes=(256, 128, 64)) ; arrêt : tour 2 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (42 s)
CatBoost, tour 1 : 54 réglages, 40 admis ; meilleur admis 88.58 % (auto_class_weights=SqrtBalanced, depth=4, l2_leaf_reg=5, learning_rate=0.03) (74 s)
CatBoost, tour 2 : 54 réglages, 37 admis ; meilleur admis 88.66 % (auto_class_weights=SqrtBalanced, depth=4, l2_leaf_reg=3, learning_rate=0.03) (73 s)
CatBoost, tour 3 : 54 réglages, 35 admis ; meilleur admis 88.77 % (auto_class_weights=Balanced, depth=5, l2_leaf_reg=1, learning_rate=0.02) (73 s)
CatBoost, tour 4 : 36 réglages, 21 admis ; meilleur admis 88.77 % (auto_class_weights=Balanced, depth=5, l2_leaf_reg=1, learning_rate=0.02) (63 s

,Réglage,"Précision moyenne sur les sorties, validation (%)",Train (%),Surapprentissage (% relatif),Alerte,Tours,Arrêt,Durée (s)
Modèle,,,,,,,,
MLP,"alpha=0.0001, hidden_layer_sizes=(256, 128, 64)",88.15 ± 1.27,92.50,4.70,—,2,tour 2 : meilleur réglage stable (à l'intérieu...,42
CatBoost,"auto_class_weights=Balanced, depth=5, l2_leaf_...",88.77 ± 1.42,95.15,6.71,—,4,tour 4 : meilleur réglage stable (à l'intérieu...,283


**Comment lire** : la précision moyenne sur les sorties vaut la part de sorties en septembre pour un tri au hasard, et 100 % pour un tri parfait (toutes les vraies sorties en tête de liste). Elle ne dépend d'aucun seuil.

## 5. Seuil à 80 % de précision marginale : sorties annoncées, pli par pli

Sur chaque pli : le modèle (meilleur réglage) est réentraîné sur les 4 autres plis. Une **validation croisée interne** à ces 4 plis (5 plis internes, par client) donne des probabilités hors pli, sur lesquelles le seuil est fixé : on descend dans la liste tant que, parmi les 50 derniers annoncés, au moins 80 % sont de vraies sorties. Le pli de validation est ensuite jugé avec ce seuil, qu'il n'a pas servi à fixer.

**Mesures sur le pli de validation** (clients en retard en août) : sorties annoncées, **taux de sorties retrouvées** (part des vraies sorties annoncées), **précision des sorties annoncées** (part de vraies sorties parmi les annoncées), précision moyenne et ROC AUC. La **précision marginale en validation** se mesure sur les 50 annoncés les plus proches de leur seuil, tous plis réunis (un pli seul en compte trop peu).

In [5]:
from sklearn.inspection import permutation_importance

PRECISION_MARGINALE_VISEE = 0.80      # fixée à l'avance par l'utilisateur
NIVEAUX = [0.70, 0.80, 0.90]          # 70 % et 90 % : pour information seulement
FENETRE_MARGE = 50                    # précision marginale mesurée sur les 50 derniers annoncés

def seuil_marginal(y, proba, visee, fenetre=FENETRE_MARGE):
    # Clients rangés du plus probablement sorti au moins probable ; on annonce tant que la part de vraies sorties
    # parmi les `fenetre` derniers annoncés reste >= visee. Renvoie la probabilité du dernier annoncé (inf : aucun)
    ordre = np.argsort(-proba, kind="mergesort")
    y_tri, p_tri = np.asarray(y)[ordre], proba[ordre]
    if len(y_tri) < fenetre:
        return np.inf
    glissante = np.convolve(y_tri, np.ones(fenetre) / fenetre, mode="valid")   # glissante[j] : part de sorties des positions j à j + fenetre - 1
    sous = np.flatnonzero(glissante < visee - 1e-9)
    if len(sous) == 0:
        return p_tri[-1]
    if sous[0] == 0:
        return np.inf
    return p_tri[sous[0] + fenetre - 2]   # dernière position de la dernière fenêtre encore au niveau visé

def probas_internes(modele, apprentissage):
    # Validation croisée interne, par client, sur les exemples d'apprentissage d'un pli :
    # probabilités hors pli des exemples de septembre de ces clients (sert à fixer le seuil)
    sous = emp.iloc[apprentissage]
    sept_sous = np.flatnonzero(sous["mois"].to_numpy() == 1)
    clients = sous["client"].to_numpy()
    y_sous = sous["sortie"].to_numpy()
    p = np.full(len(sept_sous), np.nan)
    for _, val in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(sept_sous, y_sous[sept_sous]):
        app = np.flatnonzero(~np.isin(clients, np.unique(clients[sept_sous[val]])))
        m = clone(modele).fit(sous.iloc[app][VARIABLES], y_sous[app])
        p[val] = m.predict_proba(sous.iloc[sept_sous[val]][VARIABLES])[:, 1]
    return y_sous[sept_sous], p

probas, seuils, mesures, importances = {}, {}, {}, {}
for nom, params in meilleurs.items():
    debut = time.perf_counter()
    modele = clone(ESPACES[nom]["modele"]).set_params(**params)
    p_val = np.full(len(emp), np.nan)
    seuils[nom] = {v: [] for v in NIVEAUX}
    mesures[nom] = {v: {"Annoncées": [], "Retrouvées": [], "Précision": []} for v in NIVEAUX}
    mesures[nom]["Précision moyenne"], mesures[nom]["ROC AUC"], par_pli = [], [], []
    for apprentissage, validation in cv:
        y_int, p_int = probas_internes(modele, apprentissage)
        m = clone(modele).fit(X_emp.iloc[apprentissage], y_emp.iloc[apprentissage])
        p = m.predict_proba(X_emp.iloc[validation])[:, 1]
        p_val[validation] = p
        y_v = y_emp.to_numpy()[validation]
        for v in NIVEAUX:
            s = seuil_marginal(y_int, p_int, v)
            seuils[nom][v].append(s)
            annonce = p >= s
            mesures[nom][v]["Annoncées"].append(int(annonce.sum()))
            mesures[nom][v]["Retrouvées"].append(y_v[annonce].sum() / y_v.sum())
            mesures[nom][v]["Précision"].append(y_v[annonce].mean() if annonce.any() else np.nan)
        mesures[nom]["Précision moyenne"].append(average_precision_score(y_v, p))
        mesures[nom]["ROC AUC"].append(roc_auc_score(y_v, p))
        r = permutation_importance(m, X_emp.iloc[validation], y_v, scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1)
        par_pli.append(r.importances_mean * 100)
    probas[nom] = p_val
    importances[nom] = pd.Series(np.mean(par_pli, axis=0), index=VARIABLES)
    print(f"{nom} : fait ({round(time.perf_counter() - debut)} s)", flush=True)

def moy_et(valeurs, facteur=100, decimales=1):
    v = np.asarray(valeurs, dtype=float)
    return f"{np.nanmean(v) * facteur:.{decimales}f} ± {np.nanstd(v) * facteur:.{decimales}f}"

y_sept = y_emp.to_numpy()[sept]
lignes = []
for nom in meilleurs:
    for v in NIVEAUX:
        # Précision marginale en validation : les 50 annoncés les plus proches de leur seuil, tous plis réunis
        ecarts, vrais = [], []
        for k, (_, validation) in enumerate(cv):
            p, s = probas[nom][validation], seuils[nom][v][k]
            annonce = p >= s
            ecarts += list(p[annonce] - s)
            vrais += list(y_emp.to_numpy()[validation][annonce])
        ordre = np.argsort(ecarts)[:FENETRE_MARGE]
        lignes.append({"Modèle": nom, "Précision marginale visée": f"{v:.0%}" + ("" if v == PRECISION_MARGINALE_VISEE else " (information)"),
                       "Seuil (min - max)": f"{min(seuils[nom][v]):.3f} - {max(seuils[nom][v]):.3f}",
                       "Sorties annoncées par pli": moy_et(mesures[nom][v]["Annoncées"], 1, 1),
                       "Sorties retrouvées (%)": moy_et(mesures[nom][v]["Retrouvées"]),
                       "Précision des sorties annoncées (%)": moy_et(mesures[nom][v]["Précision"]),
                       f"Précision marginale en validation, {FENETRE_MARGE} annoncés les plus proches du seuil (%)":
                           round(np.mean(np.asarray(vrais)[ordre]) * 100, 1) if len(ordre) else np.nan})
display(Markdown(f"**Sorties annoncées en validation** (moyenne ± écart-type sur les 5 plis ; {len(sept)} clients en retard en août, dont {int(y_sept.sum())} sorties)"))
display(pd.DataFrame(lignes).set_index(["Modèle", "Précision marginale visée"]))
display(pd.DataFrame({nom: {"Précision moyenne sur les sorties (%)": moy_et(mesures[nom]["Précision moyenne"]),
                            "ROC AUC": moy_et(mesures[nom]["ROC AUC"], 1, 3)} for nom in meilleurs}).T)

MLP : fait (203 s)
CatBoost : fait (46 s)


**Sorties annoncées en validation** (moyenne ± écart-type sur les 5 plis ; 2071 clients en retard en août, dont 366 sorties)

Seuil (min - max)  \
Modèle   Précision marginale visée                     
MLP      70% (information)             0.514 - 0.653   
         80%                           0.563 - 0.701   
         90% (information)             0.895 - 0.964   
CatBoost 70% (information)             0.643 - 0.668   
         80%                           0.668 - 0.824   
         90% (information)             0.892 - 0.950   

                                   Sorties annoncées par pli  \
Modèle   Précision marginale visée                             
MLP      70% (information)                        63.0 ± 6.5   
         80%                                      59.8 ± 4.4   
         90% (information)                       22.0 ± 13.1   
CatBoost 70% (information)                        67.0 ± 6.0   
         80%                                      63.4 ± 7.5   
         90% (information)                        43.8 ± 5.2   

                                   Sorties retrouvées (%)  \
Modèle   Précision marginale visée                          
MLP      70% (information)                     76.0 ± 4.8   
         80%                                   73.8 ± 4.1   
         90% (information)                    28.4 ± 16.5   
CatBoost 70% (information)                     80.9 ± 3.8   
         80%                                   77.6 ± 6.5   
         90% (information)                     55.5 ± 4.8   

                                   Précision des sorties annoncées (%)  \
Modèle   Précision marginale visée                                       
MLP      70% (information)                                  88.7 ± 4.1   
         80%                                                90.5 ± 2.9   
         90% (information)                                  96.4 ± 3.2   
CatBoost 70% (information)                                  88.7 ± 4.7   
         80%                                                90.0 ± 3.8   
         90% (information)                                  93.1 ± 3.6   

                                    Précision marginale en validation, 50 annoncés les plus proches du seuil (%)  
Modèle   Précision marginale visée                                                                                
MLP      70% (information)                                                       68.0                             
         80%                                                                     82.0                             
         90% (information)                                                       94.0                             
CatBoost 70% (information)                                                       68.0                             
         80%                                                                     78.0                             
         90% (information)                                                       94.0

,Précision moyenne sur les sorties (%),ROC AUC
MLP,88.1 ± 1.3,0.957 ± 0.006
CatBoost,88.8 ± 1.4,0.960 ± 0.004


**Comment lire** :
- **Sorties retrouvées** : sur 100 vraies sorties du retard, combien le modèle en annonce. **Précision des sorties annoncées** : sur 100 sorties annoncées, combien sont vraies. Avec une règle à la marge, la précision de l'ensemble est plus haute que 80 % : les premiers annoncés sont presque tous de vraies sorties.
- **Précision marginale en validation** : elle doit tourner autour de 80 % si le seuil s'arrête au bon endroit ; elle est mesurée sur 50 clients seulement, donc bruitée (de l'ordre de ± 6 points).
- **Un seuil qui varie beaucoup d'un pli à l'autre** (colonne « min - max ») signale un compromis instable.
- Les lignes 70 % et 90 % montrent ce que coûterait ou rapporterait un autre niveau ; elles ne servent pas à le choisir.

## 6. MLP face à CatBoost, pli par pli (précision marginale de 80 %)

Pour chaque mesure, l'écart du MLP sur CatBoost, pli par pli. Verdict de l'essai : « gain réel » si l'écart moyen dépasse l'écart-type des écarts, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [6]:
def verdict(ecarts):
    moyenne, ecart_type = np.nanmean(ecarts), np.nanstd(ecarts)
    return "✅ MLP devant" if moyenne > ecart_type else ("❌ CatBoost devant" if moyenne < -ecart_type else "➖ dans le bruit")

v = PRECISION_MARGINALE_VISEE
lignes = []
for mesure, a, b in [("Sorties retrouvées", mesures["MLP"][v]["Retrouvées"], mesures["CatBoost"][v]["Retrouvées"]),
                     ("Précision des sorties annoncées", mesures["MLP"][v]["Précision"], mesures["CatBoost"][v]["Précision"]),
                     ("Précision moyenne sur les sorties", mesures["MLP"]["Précision moyenne"], mesures["CatBoost"]["Précision moyenne"])]:
    ecarts = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    lignes.append({"Mesure": mesure, "MLP (%)": round(np.nanmean(a) * 100, 1), "CatBoost (%)": round(np.nanmean(b) * 100, 1),
                   "Écart MLP - CatBoost (points)": f"{np.nanmean(ecarts) * 100:+.1f} ± {np.nanstd(ecarts) * 100:.1f}",
                   "Plis où le MLP est devant": f"{int((ecarts > 0).sum())} sur 5", "Verdict": verdict(ecarts)})
display(pd.DataFrame(lignes).set_index("Mesure"))

# Accord des deux modèles sur les clients de septembre (prédictions hors pli, seuil de chaque pli)
annonce = {}
for nom in meilleurs:
    a = np.zeros(len(emp), dtype=bool)
    for k, (_, validation) in enumerate(cv):
        a[validation] = probas[nom][validation] >= seuils[nom][v][k]
    annonce[nom] = a[sept]
lignes = []
for libelle_groupe, masque in [("Annoncées par les deux", annonce["MLP"] & annonce["CatBoost"]),
                               ("Par le MLP seul", annonce["MLP"] & ~annonce["CatBoost"]),
                               ("Par CatBoost seul", ~annonce["MLP"] & annonce["CatBoost"]),
                               ("Par aucun des deux", ~annonce["MLP"] & ~annonce["CatBoost"])]:
    lignes.append({"Sorties annoncées": libelle_groupe, "Clients": int(masque.sum()), "Vraies sorties": int(y_sept[masque].sum()),
                   "Part de vraies sorties (%)": round(y_sept[masque].mean() * 100, 1) if masque.any() else np.nan})
display(Markdown(f"**Accord des deux modèles** ({len(sept)} clients en retard en août, {int(y_sept.sum())} sorties)"))
display(pd.DataFrame(lignes).set_index("Sorties annoncées"))

,MLP (%),CatBoost (%),Écart MLP - CatBoost (points),Plis où le MLP est devant,Verdict
Mesure,,,,,
Sorties retrouvées,73.8,77.6,-3.8 ± 5.3,1 sur 5,➖ dans le bruit
Précision des sorties annoncées,90.5,90.0,+0.5 ± 3.2,2 sur 5,➖ dans le bruit
Précision moyenne sur les sorties,88.1,88.8,-0.6 ± 0.4,1 sur 5,❌ CatBoost devant


**Accord des deux modèles** (2071 clients en retard en août, 366 sorties)

,Clients,Vraies sorties,Part de vraies sorties (%)
Sorties annoncées,,,
Annoncées par les deux,289,264,91.3
Par le MLP seul,10,6,60.0
Par CatBoost seul,28,20,71.4
Par aucun des deux,1744,76,4.4


**Comment lire** : le premier tableau dit lequel des deux modèles, à la même règle, retrouve le plus de sorties et avec quelle précision. Le second dit s'ils annoncent les mêmes clients : beaucoup de sorties annoncées par un seul des deux, avec une bonne part de vraies sorties, voudrait dire qu'ils se complètent.

## 7. L'importance des variables

Importance par permutation, sur chaque pli de validation (septembre), avec le modèle du pli : baisse de la précision moyenne sur les sorties, en points, quand la variable est mélangée (5 mélanges, moyenne sur les plis). Le second tableau regroupe les variables par ancienneté.

In [7]:
table = pd.DataFrame(importances)
display(Markdown("**Par variable**"))
display(table.sort_values("MLP", ascending=False).round(2))
def anciennete(variable):
    return "plafond" if variable == "LIMIT_BAL" else ("M" if variable.endswith("0") else f"M-{variable[-1]}")
display(Markdown("**Par ancienneté** (M = mois cible, M-1 = mois précédent, …)"))
display(table.groupby(table.index.map(anciennete)).sum().reindex(["M", "M-1", "M-2", "M-3", "M-4", "M-5", "plafond"]).round(2))

**Par variable**

,MLP,CatBoost
solde_m0,7.87,1.13
solde_m1,6.99,10.10
paiement_m0,3.45,6.16
paiement_m1,2.04,1.55
solde_m2,1.74,0.54
paiement_m2,1.65,0.85
LIMIT_BAL,1.37,0.85
codif_m3,1.22,1.89
solde_m5,0.85,1.94
codif_m4,0.85,0.96


**Par ancienneté** (M = mois cible, M-1 = mois précédent, …)

,MLP,CatBoost
M,11.32,7.29
M-1,9.04,11.65
M-2,4.11,3.02
M-3,1.91,3.24
M-4,0.84,1.59
M-5,2.04,2.53
plafond,1.37,0.85


**Comment lire** : proche de 0, le modèle ne s'en sert pas. `codif_m1` vaut 2 pour tous les exemples (ils sont tous en retard le mois précédent) : son importance est nulle par construction.

## 8. Application aux `PAY_1` = 1 en retard en août

Chaque modèle est appris sur **tous les exemples** (trois mois) avec son meilleur réglage ; son seuil est fixé sur les probabilités hors pli de septembre (section 5), à 80 % de précision marginale. Il est ensuite appliqué aux clients codifiés 1 en septembre et en retard en août. Aucun fichier de données n'est modifié ; les modèles et leurs seuils sont enregistrés dans `modeles/11c/`.

In [8]:
import json, platform
from datetime import datetime
import catboost, sklearn
from joblib.externals import cloudpickle

exemples_1 = exemples_du_mois(clients_1, 1)
exemples_1 = exemples_1[exemples_1["codif_m1"] == 2]
dossier_modeles = Path.cwd() / "modeles" / "11c"
dossier_modeles.mkdir(parents=True, exist_ok=True)
FICHIERS = {"MLP": "mlp", "CatBoost": "catboost"}
synthese = {"scenario": "11c", "date": datetime.now().isoformat(timespec="seconds"),
            "source": "data/creditcard_pret_ingestion.csv (niveau 0), sans dpnm, sans ID, sans données personnelles, sans correctifs",
            "exemples": "clients en retard le mois précédent ; mois cibles septembre, août, juillet ; notés sur septembre",
            "cible": "1 = sortie du retard (codification du mois -2, -1 ou 0) ; 0 = retard maintenu",
            "variables": VARIABLES, "critere": "précision moyenne sur les sorties (septembre)",
            "regle_de_decision": f"précision marginale de {PRECISION_MARGINALE_VISEE:.0%} sur les {FENETRE_MARGE} derniers annoncés",
            "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "catboost": catboost.__version__, "pandas": pd.__version__},
            "modeles": {}}
annonce_1, lignes = {}, []
for nom, params in meilleurs.items():
    seuil = seuil_marginal(y_sept, probas[nom][sept], PRECISION_MARGINALE_VISEE)
    modele = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_emp, y_emp)
    with open(dossier_modeles / f"11c_{FICHIERS[nom]}.pkl", "wb") as f:
        cloudpickle.dump(modele, f)
    p1 = modele.predict_proba(exemples_1[VARIABLES])[:, 1]
    annonce_1[nom] = p1 >= seuil
    synthese["modeles"][nom] = {"fichier": f"11c_{FICHIERS[nom]}.pkl", "seuil": float(seuil),
                                "reglage": {k.split("__")[-1]: (v if isinstance(v, (int, float, str)) or v is None else str(v)) for k, v in params.items()}}
    lignes.append({"Modèle": nom, "Seuil": round(float(seuil), 3),
                   "Septembre du train : part annoncée sortie (%)": round((probas[nom][sept] >= seuil).mean() * 100, 1),
                   f"PAY_1 = 1 en retard en août ({len(exemples_1)}) : annoncés sortis": int(annonce_1[nom].sum()),
                   "Part annoncée sortie (%)": round(annonce_1[nom].mean() * 100, 1),
                   "Probabilité de sortie médiane, train": round(float(np.median(probas[nom][sept])), 3),
                   "Probabilité de sortie médiane, PAY_1 = 1": round(float(np.median(p1)), 3)})
(dossier_modeles / "11c_synthese.json").write_text(json.dumps(synthese, ensure_ascii=False, indent=1), encoding="utf-8")
display(pd.DataFrame(lignes).set_index("Modèle"))
accord = pd.Series({"Annoncés sortis par les deux": int((annonce_1["MLP"] & annonce_1["CatBoost"]).sum()),
                    "Par le MLP seul": int((annonce_1["MLP"] & ~annonce_1["CatBoost"]).sum()),
                    "Par CatBoost seul": int((~annonce_1["MLP"] & annonce_1["CatBoost"]).sum()),
                    "Par aucun des deux (retard maintenu)": int((~annonce_1["MLP"] & ~annonce_1["CatBoost"]).sum())}, name="Clients")
display(accord.to_frame())
print(f"Modèles et seuils enregistrés dans {dossier_modeles}")

,Seuil,Septembre du train : part annoncée sortie (%),PAY_1 = 1 en retard en août (1824) : annoncés sortis,Part annoncée sortie (%),"Probabilité de sortie médiane, train","Probabilité de sortie médiane, PAY_1 = 1"
Modèle,,,,,,
MLP,0.630,14.5,140,7.7,0.020,0.032
CatBoost,0.724,15.5,184,10.1,0.058,0.114


,Clients
Annoncés sortis par les deux,132
Par le MLP seul,8
Par CatBoost seul,52
Par aucun des deux (retard maintenu),1632


Modèles et seuils enregistrés dans c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\modeles\11c


**Comment lire** :
- **Part annoncée sortie** : à comparer entre le septembre du train et les `PAY_1` = 1. Un écart net dirait que les 1 ne ressemblent pas aux clients tranchés : la précision mesurée en section 5 serait alors moins sûre pour eux.
- **Accord des deux modèles** : les clients annoncés sortis par les deux sont les estimations les plus solides ; ceux annoncés par un seul sont les cas douteux.
- Les clients codifiés 1 et sains en août ne sont pas traités ici : la règle les garde sains.

## 9. Précision marginale de 50 % et arbitre, pour un vote à 2 voix sur 3

*Ajout du 10/10/2026 ; les sections 1 à 8 sont inchangées (règle à 80 %).*

**Pourquoi** : la règle à 80 % de précision marginale avec l'accord du MLP et de CatBoost laisse beaucoup de codifications 1 en retard maintenu. Décisions de l'utilisateur : **précision marginale de 50 % pour chaque modèle** (on annonce une sortie tant que, parmi les 50 derniers annoncés, au moins 1 sur 2 est une vraie sortie : la sortie y est au moins aussi probable que le maintien), et un **troisième modèle d'une autre famille** comme arbitre, pour comparer la règle « deux sur deux » (MLP et CatBoost d'accord) à la règle « deux sur trois ».

**Étapes** :
1. seuils à 50 % pour le MLP et CatBoost, fixés dans des plis internes comme en section 5 (les probabilités hors pli de la section 5 sont reprises) ;
2. quatre candidats arbitres, d'autres familles que le réseau de neurones et les arbres, réglés par la même boucle (section 4) et sur le même critère : **régression logistique** (linéaire), **SVM** à noyau gaussien (frontière souple ; il donne un score et non une probabilité, le seuil se calcule sur ce score de la même façon), **KNN** (voisinage), **bayésien naïf** gaussien (probabiliste) ; l'arbitre retenu est celui qui a la meilleure précision moyenne sur les sorties en validation ;
3. comparaison des règles sur le septembre du train, puis nombre de sorties estimées chez les `PAY_1` = 1 en retard en août, pour chaque règle.

### 9.1 Seuils à 50 % pour le MLP et CatBoost

In [9]:
PRECISION_50 = 0.50

def score(modele, X):
    # Probabilité de sortie, ou score pour un modèle qui ne donne pas de probabilité (SVM) : seul l'ordre des clients compte
    return modele.predict_proba(X)[:, 1] if hasattr(modele, "predict_proba") else modele.decision_function(X)

def scores_internes(modele, apprentissage):
    # Validation croisée interne, par client, sur les exemples d'apprentissage d'un pli : scores hors pli de septembre
    sous = emp.iloc[apprentissage]
    sept_sous = np.flatnonzero(sous["mois"].to_numpy() == 1)
    clients, y_sous = sous["client"].to_numpy(), sous["sortie"].to_numpy()
    s = np.full(len(sept_sous), np.nan)
    for _, val in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(sept_sous, y_sous[sept_sous]):
        app = np.flatnonzero(~np.isin(clients, np.unique(clients[sept_sous[val]])))
        s[val] = score(clone(modele).fit(sous.iloc[app][VARIABLES], y_sous[app]), sous.iloc[sept_sous[val]][VARIABLES])
    return y_sous[sept_sous], s

def annonces_a_50(nom, modele):
    # Seuil à 50 % de précision marginale fixé dans les plis internes, appliqué aux scores hors pli du pli de validation
    a = np.zeros(len(emp), dtype=bool)
    seuils_50 = []
    for apprentissage, validation in cv:
        y_int, s_int = scores_internes(modele, apprentissage)
        s = seuil_marginal(y_int, s_int, PRECISION_50)
        seuils_50.append(s)
        a[validation] = probas[nom][validation] >= s
    return a[sept], seuils_50

annonce_50, seuils_a_50 = {}, {}
for nom in ["MLP", "CatBoost"]:
    debut = time.perf_counter()
    annonce_50[nom], seuils_a_50[nom] = annonces_a_50(nom, clone(ESPACES[nom]["modele"]).set_params(**meilleurs[nom]))
    print(f"{nom} : seuils {min(seuils_a_50[nom]):.3f} - {max(seuils_a_50[nom]):.3f} ; sorties annoncées {int(annonce_50[nom].sum())}, "
          f"précision {y_sept[annonce_50[nom]].mean() * 100:.1f} %, retrouvées {y_sept[annonce_50[nom]].sum() / y_sept.sum() * 100:.1f} % "
          f"({round(time.perf_counter() - debut)} s)", flush=True)

MLP : seuils 0.373 - 0.459 ; sorties annoncées 367, précision 82.0 %, retrouvées 82.2 % (183 s)
CatBoost : seuils 0.493 - 0.570 ; sorties annoncées 369, précision 82.7 %, retrouvées 83.3 % (35 s)


**Comment lire** : pour chaque modèle, l'étendue de ses seuils sur les 5 plis, le nombre de sorties annoncées sur le septembre du train (2 071 clients en retard en août), leur précision et la part des vraies sorties retrouvées. À comparer avec la section 5 (seuil à 80 %).

### 9.2 Les quatre candidats arbitres

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

def score_septembre(estimateur, X, y):
    # Précision moyenne sur les sorties, sur les seuls exemples de septembre (codif_m5 renseignée) ; accepte un score (SVM)
    septembre = X["codif_m5"].notna().to_numpy()
    return average_precision_score(np.asarray(y)[septembre], score(estimateur, X[septembre]))

def pipeline_lineaire(estimateur):
    # Variables comme pour le MLP : codifications en une colonne par valeur, montants au logarithme signé puis mis à l'échelle, vides à 0 avec indicateur
    return make_pipeline(FunctionTransformer(log_montants), preprocesseur_lineaire(), estimateur)

CANDIDATS = {
    "Régression logistique": {"modele": pipeline_lineaire(LogisticRegression(solver="saga", max_iter=5000)),
                              "echelles": {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]},
                              "depart": {"logisticregression__C": 1},
                              "choix": {"logisticregression__class_weight": [None, "balanced"], "logisticregression__l1_ratio": [0, 0.5, 1]}},
    "SVM": {"modele": pipeline_lineaire(SVC(kernel="rbf")),
            "echelles": {"svc__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100], "svc__gamma": [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3]},
            "depart": {"svc__C": 1, "svc__gamma": 0.01},
            "choix": {"svc__class_weight": [None, "balanced"]}},
    "KNN": {"modele": pipeline_lineaire(KNeighborsClassifier(n_jobs=1)),
            "echelles": {"kneighborsclassifier__n_neighbors": [5, 10, 20, 30, 50, 75, 100, 150, 200, 300]},
            "depart": {"kneighborsclassifier__n_neighbors": 50},
            "choix": {"kneighborsclassifier__weights": ["uniform", "distance"]}},
    "Bayésien naïf": {"modele": pipeline_lineaire(GaussianNB()),
                      "echelles": {"gaussiannb__var_smoothing": [1e-12, 1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
                      "depart": {"gaussiannb__var_smoothing": 1e-9},
                      "choix": {}},
}

lignes, reglages_candidats = [], {}
for nom, espace in CANDIDATS.items():
    historique, raison, duree = optimiser(nom, espace)   # boucle de la section 4
    admis = historique[historique["admis"]]
    if admis.empty:
        ligne, alerte, valeur = historique.loc[historique["surapprentissage (% relatif)"].idxmin()], "⚠️ aucun réglage admis", -np.inf
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
        valeur = ligne["validation"]
    reglages_candidats[nom] = (ligne["params"], valeur)
    lignes.append({"Candidat": nom, "Réglage": libelle(ligne["params"]),
                   "Précision moyenne sur les sorties, validation (%)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2), "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Arrêt": raison, "Durée (s)": duree})
display(pd.DataFrame(lignes).set_index("Candidat"))
display(pd.DataFrame({nom: {"Précision moyenne sur les sorties (%)": moy_et(mesures[nom]["Précision moyenne"])} for nom in ["MLP", "CatBoost"]}).T
        .rename_axis("Pour comparaison (section 5)"))

ARBITRE = max(reglages_candidats, key=lambda nom: reglages_candidats[nom][1])
modele_arbitre = clone(CANDIDATS[ARBITRE]["modele"]).set_params(**reglages_candidats[ARBITRE][0])
print(f"Arbitre retenu : {ARBITRE} ({libelle(reglages_candidats[ARBITRE][0])})")

# Scores hors pli de l'arbitre (mêmes plis), puis son seuil à 50 %
debut = time.perf_counter()
probas[ARBITRE] = np.full(len(emp), np.nan)
for apprentissage, validation in cv:
    probas[ARBITRE][validation] = score(clone(modele_arbitre).fit(X_emp.iloc[apprentissage], y_emp.iloc[apprentissage]), X_emp.iloc[validation])
annonce_50[ARBITRE], seuils_a_50[ARBITRE] = annonces_a_50(ARBITRE, modele_arbitre)
print(f"{ARBITRE} : sorties annoncées {int(annonce_50[ARBITRE].sum())}, précision {y_sept[annonce_50[ARBITRE]].mean() * 100:.1f} %, "
      f"retrouvées {y_sept[annonce_50[ARBITRE]].sum() / y_sept.sum() * 100:.1f} % ({round(time.perf_counter() - debut)} s)")

Régression logistique, tour 1 : 18 réglages, 18 admis ; meilleur admis 80.16 % (C=0.3, class_weight=None, l1_ratio=1) (4 s)
Régression logistique, tour 2 : 18 réglages, 18 admis ; meilleur admis 80.46 % (C=0.1, class_weight=None, l1_ratio=1) (2 s)
Régression logistique, tour 3 : 18 réglages, 18 admis ; meilleur admis 80.46 % (C=0.1, class_weight=None, l1_ratio=1) (1 s)
SVM, tour 1 : 18 réglages, 18 admis ; meilleur admis 88.00 % (C=1, class_weight=balanced, gamma=0.03) (8 s)
SVM, tour 2 : 18 réglages, 14 admis ; meilleur admis 88.00 % (C=1, class_weight=balanced, gamma=0.03) (8 s)
KNN, tour 1 : 6 réglages, 3 admis ; meilleur admis 82.70 % (n_neighbors=30, weights=uniform) (0 s)
KNN, tour 2 : 6 réglages, 3 admis ; meilleur admis 82.70 % (n_neighbors=30, weights=uniform) (0 s)
Bayésien naïf, tour 1 : 3 réglages, 3 admis ; meilleur admis 68.31 % (var_smoothing=1e-09) (0 s)


,Réglage,"Précision moyenne sur les sorties, validation (%)",Train (%),Surapprentissage (% relatif),Alerte,Arrêt,Durée (s)
Candidat,,,,,,,
Régression logistique,"C=0.1, class_weight=None, l1_ratio=1",80.46 ± 2.07,80.30,-0.20,⚠️ variance (2.1 points),tour 3 : meilleur réglage stable (à l'intérieu...,8
SVM,"C=1, class_weight=balanced, gamma=0.03",88.00 ± 2.04,90.33,2.58,⚠️ variance (2.0 points),tour 2 : meilleur réglage stable (à l'intérieu...,16
KNN,"n_neighbors=30, weights=uniform",82.70 ± 1.53,85.43,3.19,—,tour 2 : meilleur réglage stable (à l'intérieu...,1
Bayésien naïf,var_smoothing=1e-09,68.31 ± 2.32,68.25,-0.09,⚠️ variance (2.3 points),tour 1 : meilleur réglage stable (à l'intérieu...,0


,Précision moyenne sur les sorties (%)
Pour comparaison (section 5),
MLP,88.1 ± 1.3
CatBoost,88.8 ± 1.4


Arbitre retenu : SVM (C=1, class_weight=balanced, gamma=0.03)
SVM : sorties annoncées 376, précision 81.1 %, retrouvées 83.3 % (25 s)


**Comment lire** : une ligne par candidat, avec son meilleur réglage admis (surapprentissage de 7 % relatif au plus) et sa précision moyenne sur les sorties en validation ; le second tableau rappelle celle du MLP et de CatBoost. Le candidat retenu comme arbitre est celui dont la précision moyenne est la plus haute.

### 9.3 Les règles comparées

In [11]:
from joblib.externals import cloudpickle

voix = sum(annonce_50[nom].astype(int) for nom in ["MLP", "CatBoost", ARBITRE])
REGLES = {"MLP seul": annonce_50["MLP"], "CatBoost seul": annonce_50["CatBoost"], f"{ARBITRE} seul": annonce_50[ARBITRE],
          "Deux sur deux (MLP et CatBoost d'accord)": annonce_50["MLP"] & annonce_50["CatBoost"],
          f"Deux sur trois (MLP, CatBoost, {ARBITRE})": voix >= 2,
          "Rappel : deux sur deux à 80 % (règle actuelle)": annonce["MLP"] & annonce["CatBoost"]}

# Application aux PAY_1 = 1 en retard en août (section 8) : modèles appris sur tous les exemples, seuils à 50 % sur les scores hors pli
finaux = {}
for nom in ["MLP", "CatBoost"]:
    with open(dossier_modeles / f"11c_{FICHIERS[nom]}.pkl", "rb") as f:
        finaux[nom] = cloudpickle.load(f)
finaux[ARBITRE] = clone(modele_arbitre).fit(X_emp, y_emp)
annonce_1_50 = {nom: score(m, exemples_1[VARIABLES]) >= seuil_marginal(y_sept, probas[nom][sept], PRECISION_50) for nom, m in finaux.items()}
voix_1 = sum(a.astype(int) for a in annonce_1_50.values())
REGLES_1 = {"MLP seul": annonce_1_50["MLP"], "CatBoost seul": annonce_1_50["CatBoost"], f"{ARBITRE} seul": annonce_1_50[ARBITRE],
            "Deux sur deux (MLP et CatBoost d'accord)": annonce_1_50["MLP"] & annonce_1_50["CatBoost"],
            f"Deux sur trois (MLP, CatBoost, {ARBITRE})": voix_1 >= 2,
            "Rappel : deux sur deux à 80 % (règle actuelle)": annonce_1["MLP"] & annonce_1["CatBoost"]}

lignes = []
for libelle_regle, a in REGLES.items():
    lignes.append({"Règle": libelle_regle, "Sorties annoncées": int(a.sum()), "Vraies sorties parmi elles": int(y_sept[a].sum()),
                   "Précision des sorties annoncées (%)": round(y_sept[a].mean() * 100, 1) if a.any() else np.nan,
                   "Sorties retrouvées (%)": round(y_sept[a].sum() / y_sept.sum() * 100, 1),
                   f"PAY_1 = 1 en retard en août ({len(exemples_1)}) : sorties estimées": int(REGLES_1[libelle_regle].sum()),
                   "Part (%)": round(REGLES_1[libelle_regle].mean() * 100, 1)})
display(Markdown(f"**Règles comparées** (septembre du train : {len(sept)} clients en retard en août, dont {int(y_sept.sum())} sorties, "
                 f"soit {y_sept.mean() * 100:.1f} % ; prédictions hors pli)"))
display(pd.DataFrame(lignes).set_index("Règle"))

# Recoupement : quels modèles annoncent quels clients (seuils à 50 %)
noms = ["MLP", "CatBoost", ARBITRE]
def combinaisons(annonces, n):
    return pd.Series([" + ".join(m for m in noms if annonces[m][i]) or "aucun" for i in range(n)])
combi_train = combinaisons(annonce_50, len(sept))
combi_1 = combinaisons(annonce_1_50, len(exemples_1))
recoupement = pd.DataFrame({"Septembre du train : clients": combi_train.value_counts(),
                            "dont vraies sorties": pd.Series(y_sept).groupby(combi_train).sum()})
recoupement["Part de vraies sorties (%)"] = (recoupement["dont vraies sorties"] / recoupement["Septembre du train : clients"] * 100).round(1)
recoupement["PAY_1 = 1 en retard en août : clients"] = combi_1.value_counts()
recoupement = recoupement.fillna(0).astype({"Septembre du train : clients": int, "dont vraies sorties": int, "PAY_1 = 1 en retard en août : clients": int})
recoupement["Voix"] = [0 if k == "aucun" else k.count("+") + 1 for k in recoupement.index]
display(Markdown("**Recoupement : quels modèles annoncent la sortie de quels clients** (seuils à 50 % de précision marginale)"))
display(recoupement.sort_values(["Voix", "Septembre du train : clients"], ascending=False).rename_axis("Sortie annoncée par"))

**Règles comparées** (septembre du train : 2071 clients en retard en août, dont 366 sorties, soit 17.7 % ; prédictions hors pli)

,Sorties annoncées,Vraies sorties parmi elles,Précision des sorties annoncées (%),Sorties retrouvées (%),PAY_1 = 1 en retard en août (1824) : sorties estimées,Part (%)
Règle,,,,,,
MLP seul,367,301,82.0,82.2,191,10.5
CatBoost seul,369,305,82.7,83.3,258,14.1
SVM seul,376,305,81.1,83.3,245,13.4
Deux sur deux (MLP et CatBoost d'accord),343,296,86.3,80.9,185,10.1
"Deux sur trois (MLP, CatBoost, SVM)",368,304,82.6,83.1,220,12.1
Rappel : deux sur deux à 80 % (règle actuelle),289,264,91.3,72.1,132,7.2


**Recoupement : quels modèles annoncent la sortie de quels clients** (seuils à 50 % de précision marginale)

,Septembre du train : clients,dont vraies sorties,Part de vraies sorties (%),PAY_1 = 1 en retard en août : clients,Voix
Sortie annoncée par,,,,,
MLP + CatBoost + SVM,337,294,87.2,185,3
MLP + SVM,17,4,23.5,4,2
CatBoost + SVM,8,4,50.0,31,2
MLP + CatBoost,6,2,33.3,0,2
CatBoost,18,5,27.8,42,1
SVM,14,3,21.4,25,1
MLP,7,1,14.3,2,1
aucun,1664,53,3.2,1535,0


**Comment lire** :
- **Septembre du train** : pour chaque règle, le nombre de sorties annoncées, leur précision (part de vraies sorties) et la part des vraies sorties retrouvées. Les règles à 50 % annoncent plus de sorties que la règle actuelle (dernière ligne), avec une précision plus basse.
- **`PAY_1` = 1 en retard en août** : le nombre de codifications 1 que chaque règle ferait sortir du retard (estimées -1), et leur part. Dans le train, 17,7 % des clients en retard en août sortent du retard.
- **Recoupement** : une ligne par groupe de modèles qui annoncent la sortie (« aucun » : aucun modèle). Beaucoup de clients annoncés par un seul modèle, avec une bonne part de vraies sorties, voudrait dire que les modèles se complètent ; si presque tout est annoncé par les trois à la fois, l'arbitre ne change presque rien. Les groupes à deux ou trois modèles forment la règle « deux sur trois ».
- **Choix de la règle** (utilisateur) : « deux sur deux » est plus prudente ; « deux sur trois » laisse l'arbitre départager le MLP et CatBoost quand ils ne sont pas d'accord. La règle choisie sera reportée dans `02_02_modele_sorties_retard` et dans `02_01_nettoyage`.

### 9.4 Enregistrement des six modèles à 50 % (`modeles/11c/`)

Pour d'autres comparatifs, les six modèles sont gardés avec leur seuil à 50 % de précision marginale : le MLP et CatBoost (déjà enregistrés en section 8, appris sur tous les exemples) et les quatre candidats arbitres, appris ici sur tous les exemples avec leur meilleur réglage. Une synthèse à part, `11c_synthese_50.json`, donne pour chacun le fichier, le réglage, le seuil à 50 % (fixé sur les scores hors pli de septembre) et l'arbitre retenu ; `11c_synthese.json` (seuils à 80 %, section 8) reste inchangée. Enregistrés avec `cloudpickle` : ils se rechargent avec `pickle.load`, sans rien redéfinir. Le SVM donne un score et non une probabilité : son seuil porte sur ce score (`decision_function`).

In [12]:
FICHIERS_50 = {"MLP": "mlp", "CatBoost": "catboost", "Régression logistique": "regression_logistique",
               "SVM": "svm", "KNN": "knn", "Bayésien naïf": "bayesien_naif"}
synthese_50 = {"scenario": "11c, section 9", "date": datetime.now().isoformat(timespec="seconds"),
               "source": synthese["source"], "exemples": synthese["exemples"], "cible": synthese["cible"], "variables": VARIABLES,
               "regle_de_decision": f"précision marginale de {PRECISION_50:.0%} sur les {FENETRE_MARGE} derniers annoncés, pour chaque modèle",
               "arbitre_retenu": ARBITRE, "versions": synthese["versions"], "modeles": {}}
lignes = []
for nom in FICHIERS_50:
    fichier = dossier_modeles / f"11c_{FICHIERS_50[nom]}.pkl"
    if nom in CANDIDATS:
        modele_final = finaux[nom] if nom == ARBITRE else clone(CANDIDATS[nom]["modele"]).set_params(**reglages_candidats[nom][0]).fit(X_emp, y_emp)
        with open(fichier, "wb") as f:
            cloudpickle.dump(modele_final, f)
        reglage = reglages_candidats[nom][0]
        s_hors_pli = probas[nom][sept] if nom in probas else np.full(len(sept), np.nan)
        if nom not in probas:   # scores hors pli des candidats non retenus, pour fixer leur seuil
            s_hors_pli = np.full(len(emp), np.nan)
            m = clone(CANDIDATS[nom]["modele"]).set_params(**reglage)
            for apprentissage, validation in cv:
                s_hors_pli[validation] = score(clone(m).fit(X_emp.iloc[apprentissage], y_emp.iloc[apprentissage]), X_emp.iloc[validation])
            probas[nom] = s_hors_pli
            s_hors_pli = s_hors_pli[sept]
    else:
        reglage = meilleurs[nom]          # MLP et CatBoost : fichiers de la section 8, inchangés
        s_hors_pli = probas[nom][sept]
    seuil = float(seuil_marginal(y_sept, s_hors_pli, PRECISION_50))
    synthese_50["modeles"][nom] = {"fichier": fichier.name, "seuil_50": seuil, "score": "decision_function" if nom == "SVM" else "predict_proba",
                                   "reglage": {k.split("__")[-1]: (v if isinstance(v, (int, float, str)) or v is None else str(v)) for k, v in reglage.items()}}
    lignes.append({"Modèle": nom, "Fichier": fichier.name, "Taille (Ko)": fichier.stat().st_size // 1024, "Seuil à 50 %": round(seuil, 3),
                   "Arbitre retenu": "oui" if nom == ARBITRE else ""})
(dossier_modeles / "11c_synthese_50.json").write_text(json.dumps(synthese_50, ensure_ascii=False, indent=1), encoding="utf-8")
display(pd.DataFrame(lignes).set_index("Modèle"))
print(f"Synthèse : {dossier_modeles / '11c_synthese_50.json'}")

,Fichier,Taille (Ko),Seuil à 50 %,Arbitre retenu
Modèle,,,,
MLP,11c_mlp.pkl,5996,0.484,
CatBoost,11c_catboost.pkl,595,0.592,
Régression logistique,11c_regression_logistique.pkl,5,0.375,
SVM,11c_svm.pkl,1386,0.107,oui
KNN,11c_knn.pkl,2506,0.367,
Bayésien naïf,11c_bayesien_naif.pkl,6,0.000,


Synthèse : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\modeles\11c\11c_synthese_50.json


**Comment lire** : une ligne par modèle enregistré, avec son fichier, sa taille et son seuil à 50 %. Les fichiers du MLP et de CatBoost sont ceux de la section 8 (les mêmes modèles) ; seul leur seuil change dans la synthèse à 50 %.

## 10. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*